# Step 4 - Stationarity Check and ADF Test

This notebook checks whether the imputed PJM East hourly demand series is stationary. The analysis combines a visual inspection of rolling statistics with formal Augmented Dickey-Fuller (ADF) and KPSS tests, then records the final decision for the next modeling step.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.tsa.stattools import adfuller, kpss

PROCESSED_PATH = "../data/processed/"
FIGURES_PATH = "../reports/figures/"
ROLLING_WINDOW = 24 * 7

os.makedirs(FIGURES_PATH, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

df = pd.read_csv(
    os.path.join(PROCESSED_PATH, "pjme_imputed.csv"),
    parse_dates=["Datetime"],
    index_col="Datetime",
).sort_index()

series = df["PJME_MW"].astype(float)

print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Missing values: {int(series.isna().sum())}")
df.head()

In [ ]:
rolling_mean = series.rolling(window=ROLLING_WINDOW).mean()
rolling_std = series.rolling(window=ROLLING_WINDOW).std()

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)

series.plot(ax=axes[0], color="steelblue", linewidth=0.5)
axes[0].set_title("PJME hourly demand")
axes[0].set_ylabel("Demand (MW)")

rolling_mean.plot(ax=axes[1], color="darkorange", linewidth=1)
axes[1].set_title("Rolling mean (window = 168 hours)")
axes[1].set_ylabel("Rolling mean (MW)")

rolling_std.plot(ax=axes[2], color="seagreen", linewidth=1)
axes[2].set_title("Rolling standard deviation (window = 168 hours)")
axes[2].set_ylabel("Rolling std (MW)")
axes[2].set_xlabel("Datetime")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, "04_rolling_stats.png"), dpi=150, bbox_inches="tight")
plt.show()

rolling_summary = pd.DataFrame(
    {
        "metric": [
            "rolling_mean_min",
            "rolling_mean_max",
            "rolling_std_min",
            "rolling_std_max",
        ],
        "value": [
            rolling_mean.dropna().min(),
            rolling_mean.dropna().max(),
            rolling_std.dropna().min(),
            rolling_std.dropna().max(),
        ],
    }
)
display(rolling_summary.style.format({"value": "{:,.2f}"}))

A stationary series should have a roughly constant mean and variance over time. Here, both the rolling mean and the rolling standard deviation move substantially across the sample, which is visual evidence that the raw PJME demand series is not stationary and still contains strong structural changes and seasonality.

In [ ]:
adf_result = adfuller(series, autolag="AIC")
adf_summary = pd.DataFrame(
    {
        "metric": ["test_statistic", "p_value", "used_lags", "n_obs", "critical_value_1pct", "critical_value_5pct", "critical_value_10pct"],
        "value": [
            adf_result[0],
            adf_result[1],
            adf_result[2],
            adf_result[3],
            adf_result[4]["1%"],
            adf_result[4]["5%"],
            adf_result[4]["10%"],
        ],
    }
)

print("ADF test")
display(adf_summary.style.format({"value": "{:,.6f}"}))

adf_conclusion = "Reject H0: evidence against a unit root" if adf_result[1] < 0.05 else "Fail to reject H0: series appears non-stationary"
print(adf_conclusion)

The ADF test rejects the null hypothesis of a unit root because the p-value is effectively zero. On its own that would suggest stationarity, but ADF can reject very easily in large samples even when the series still contains strong seasonal structure and changing mean behavior, so the ADF result should be interpreted together with the visual evidence and KPSS result rather than in isolation.

In [ ]:
kpss_result = kpss(series, regression="c", nlags="auto")
kpss_trend_result = kpss(series, regression="ct", nlags="auto")

kpss_summary = pd.DataFrame(
    [
        {
            "test": "KPSS_level_stationarity",
            "test_statistic": kpss_result[0],
            "p_value": kpss_result[1],
            "used_lags": kpss_result[2],
            "critical_value_10pct": kpss_result[3]["10%"],
            "critical_value_5pct": kpss_result[3]["5%"],
            "critical_value_1pct": kpss_result[3]["1%"],
        },
        {
            "test": "KPSS_trend_stationarity",
            "test_statistic": kpss_trend_result[0],
            "p_value": kpss_trend_result[1],
            "used_lags": kpss_trend_result[2],
            "critical_value_10pct": kpss_trend_result[3]["10%"],
            "critical_value_5pct": kpss_trend_result[3]["5%"],
            "critical_value_1pct": kpss_trend_result[3]["1%"],
        },
    ]
)

print("KPSS tests")
display(kpss_summary.style.format({
    "test_statistic": "{:,.6f}",
    "p_value": "{:,.6f}",
    "used_lags": "{:,.0f}",
    "critical_value_10pct": "{:,.3f}",
    "critical_value_5pct": "{:,.3f}",
    "critical_value_1pct": "{:,.3f}",
}))

The KPSS test uses the opposite null hypothesis: it assumes stationarity unless the data provide evidence otherwise. Both the level-stationary and trend-stationary KPSS variants reject their null hypotheses at the reported boundary p-value, which supports the visual conclusion that the series still has non-stationary behavior.

In [ ]:
overall_decision = "Proceed to Step 5: treat PJME_MW as non-stationary and test transformations."
reasoning = "Rolling mean/std vary materially over time and KPSS rejects stationarity despite ADF rejecting a unit root on the very large sample."

stationarity_results = pd.DataFrame(
    [
        {
            "test": "ADF",
            "null_hypothesis": "Series has a unit root (non-stationary)",
            "test_statistic": adf_result[0],
            "p_value": adf_result[1],
            "used_lags": adf_result[2],
            "n_obs": adf_result[3],
            "decision_at_5pct": "Reject H0",
            "interpretation": "Suggests no unit root, but not sufficient alone because seasonality and changing moments remain visible.",
            "overall_decision": overall_decision,
            "reasoning": reasoning,
        },
        {
            "test": "KPSS_level_stationarity",
            "null_hypothesis": "Series is level stationary",
            "test_statistic": kpss_result[0],
            "p_value": kpss_result[1],
            "used_lags": kpss_result[2],
            "n_obs": len(series),
            "decision_at_5pct": "Reject H0",
            "interpretation": "Suggests the raw series is not level stationary.",
            "overall_decision": overall_decision,
            "reasoning": reasoning,
        },
        {
            "test": "KPSS_trend_stationarity",
            "null_hypothesis": "Series is trend stationary",
            "test_statistic": kpss_trend_result[0],
            "p_value": kpss_trend_result[1],
            "used_lags": kpss_trend_result[2],
            "n_obs": len(series),
            "decision_at_5pct": "Reject H0",
            "interpretation": "Suggests the raw series is not stationary even around a deterministic trend.",
            "overall_decision": overall_decision,
            "reasoning": reasoning,
        },
    ]
)

stationarity_results.to_csv(os.path.join(PROCESSED_PATH, "stationarity_results.csv"), index=False)
print("Saved results to ../data/processed/stationarity_results.csv")
stationarity_results

Final decision: treat the raw `PJME_MW` series as non-stationary and continue to Step 5. Even though ADF rejects a unit root, the rolling statistics are clearly unstable and KPSS rejects stationarity, so the safer modeling decision is to apply transformations and retest before choosing final ARIMA or SARIMA orders.